In [1]:
!pip install langchain-openai langchain-chroma chromadb boto3 python-dotenv tiktoken

In [2]:
import os
import json
import boto3
import chromadb
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

In [3]:
# ==========================================
# 1. CONFIGURACIÓN Y CREDENCIALES
# ==========================================
load_dotenv()
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION")

# Configuración S3
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
S3_PREFIX_GOLD = os.getenv("S3_PREFIX_GOLD")

# Inicializar modelo de Embeddings (Azure)
embeddings_model = AzureOpenAIEmbeddings(
        azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
        openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY")
    )
# Inicializar Base de Datos Vectorial (Chroma)
# ==========================================
# CONEXIÓN A CHROMADB EN EC2
# ==========================================
host_chroma = os.getenv("CHROMA_SERVER_HOST")
port_chroma = os.getenv("CHROMA_SERVER_PORT")
COLECCION_NOMBRE = "fds_quimicos"


# 1. Creamos el cliente HTTP nativo de Chroma
chroma_client = chromadb.HttpClient(host=host_chroma, port=port_chroma)

In [4]:
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

In [5]:
vector_db = Chroma(
    client=chroma_client,
    collection_name=COLECCION_NOMBRE,
    embedding_function=embeddings_model
)

print("✅ Conexión establecida con éxito.")

✅ Conexión establecida con éxito.


In [6]:
# ==========================================
# 2. FUNCIONES DE PREPARACIÓN PARA CHROMA
# ==========================================
def aplanar_metadatos(chunk_dict):
    """
    ChromaDB solo acepta strings, ints, floats o bools en metadatos.
    Extraemos la jerarquía anidada y la ponemos al primer nivel.
    """
    meta_plano = {
        "doc_id": chunk_dict.get("doc_id", "desconocido"),
        "seccion": chunk_dict.get("seccion", 0),
        "chunk_id": chunk_dict.get("chunk_id", ""),
        "longitud_chars": chunk_dict.get("longitud_chars", 0)
    }

    # Extraer los títulos (Headers) de la jerarquía si existen
    jerarquia = chunk_dict.get("jerarquia", {})
    if "Header_1" in jerarquia: meta_plano["Header_1"] = jerarquia["Header_1"]
    if "Header_2" in jerarquia: meta_plano["Header_2"] = jerarquia["Header_2"]
    if "Header_3" in jerarquia: meta_plano["Header_3"] = jerarquia["Header_3"]

    # Limpiamos valores nulos (Chroma no acepta None)
    return {k: v for k, v in meta_plano.items() if v is not None}

In [7]:
def obtener_archivos_gold(prefix):
    paginator = s3_client.get_paginator('list_objects_v2')
    archivos = []
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                if obj['Key'].endswith('_chunks.jsonl'):
                    archivos.append(obj['Key'])
    return archivos

In [8]:
# ==========================================
# 3. ORQUESTADOR MAESTRO (VECTORIZACIÓN)
# ==========================================
def ingestar_en_chromadb():
    print(" Iniciando Pipeline: Gold -> Base Vectorial (ChromaDB)")

    # Forzamos la búsqueda en la subcarpeta 'chunks/' dinámicamente
    prefix_busqueda = f"{S3_PREFIX_GOLD}chunks/"

    archivos_gold = obtener_archivos_gold(prefix_busqueda)
    if not archivos_gold:
        print(f" No se encontraron archivos de chunks en la ruta: {prefix_busqueda}")
        return

    print(f" Encontrados {len(archivos_gold)} documentos listos para vectorizar.\n")

    for s3_key in archivos_gold:
        doc_id = os.path.basename(s3_key).replace('_chunks.jsonl', '')
        print(f" Vectorizando: {doc_id}")

        # Descargar y parsear archivo
        response = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=s3_key)
        lineas = response['Body'].read().decode('utf-8').strip().split('\n')

        documentos_langchain = []

        for linea in lineas:
            if not linea: continue
            chunk = json.loads(linea)

            # Formatear a objeto Document de LangChain
            meta_seguro = aplanar_metadatos(chunk)
            doc_lc = Document(
                page_content=chunk.get("contenido", ""),
                metadata=meta_seguro,
                id=meta_seguro["chunk_id"] # Usar nuestro ID en lugar de uno autogenerado
            )
            documentos_langchain.append(doc_lc)

        # Ingestar en ChromaDB (LangChain maneja el envío en batches hacia el LLM)
        try:
            # add_documents calcula los embeddings automáticamente usando Azure
            vector_db.add_documents(documents=documentos_langchain, ids=[doc.id for doc in documentos_langchain])
            print(f"   Éxito. {len(documentos_langchain)} vectores guardados en ChromaDB.")
        except Exception as e:
            print(f"   Error al vectorizar {doc_id}: {e}")

    print("\n VECTORIZACIÓN COMPLETADA.")
    print(f" Colección '{COLECCION_NOMBRE}' lista para recibir consultas.")

In [9]:
ingestar_en_chromadb()

 Iniciando Pipeline: Gold -> Base Vectorial (ChromaDB)
 Encontrados 23 documentos listos para vectorizar.

 Vectorizando: Esmalte Uretano AR Comp. B
   Éxito. 43 vectores guardados en ChromaDB.
 Vectorizando: FDS 1 -pintura-antideslizante-para-canchas
   Éxito. 38 vectores guardados en ChromaDB.
 Vectorizando: FDS 10 - pintura-epoxi-poliamida-blanco-13243-10012925-10344427-hoja-de-seguridad-pdf
   Éxito. 139 vectores guardados en ChromaDB.
 Vectorizando: FDS 11  - Ecosphere Premium
   Éxito. 90 vectores guardados en ChromaDB.
 Vectorizando: FDS 12 - PINTURA ANTICORROSIVA GRIS 507 _ 10014333-10012454-10171712 _COL (Versión 3) (1)
   Éxito. 144 vectores guardados en ChromaDB.
 Vectorizando: FDS 13 -AVANTEX-FACHADA
   Éxito. 43 vectores guardados en ChromaDB.
 Vectorizando: FDS 14 -PINTURA-ACRILICA-PARA-TRÁFICO
   Éxito. 40 vectores guardados en ChromaDB.
 Vectorizando: FDS 15 - impra
   Éxito. 49 vectores guardados en ChromaDB.
 Vectorizando: FDS 16 - sellador-nitrocelulosico
   Éxito. 4

In [10]:
# ==========================================
# 4. TEST DE RECUPERACIÓN SEMÁNTICA
# ==========================================
pregunta = "¿Qué equipos de protección personal (EPP) y guantes se deben utilizar?"

print(f"🔍 Buscando: '{pregunta}'")
# Buscamos los 3 chunks más relevantes
resultados = vector_db.similarity_search_with_score(pregunta, k=3)

for i, (doc, score) in enumerate(resultados):
    print(f"\n--- Top {i+1} (Score/Distancia: {score:.4f}) ---")
    print(f"📄 Documento: {doc.metadata.get('doc_id')}")
    print(f"📌 Sección: {doc.metadata.get('seccion')}")
    print(f"🔖 Subtítulo: {doc.metadata.get('Header_1', '')} > {doc.metadata.get('Header_2', '')}")
    print(f"📝 Extracto: {doc.page_content[:300]}...")

🔍 Buscando: '¿Qué equipos de protección personal (EPP) y guantes se deben utilizar?'

--- Top 1 (Score/Distancia: 0.6075) ---
📄 Documento: FDS 18 - KORAZA BLANCO 2650 _ 10015206-10016171-10017460-10128944-10253596-10287798-10346307-10346308 _COL
📌 Sección: 5
🔖 Subtítulo:  > 8.2 Controles técnicos apropiados:
📝 Extracto: - A.- Medidas de protección individual, como equipo de protección personal (EPP)  
Como medida de prevención se recomienda la utilización de equipos de protección individual básicos. Para más información sobre los equipos de protección individual (almacenamiento, uso, limpieza, mantenimiento, clase ...

--- Top 2 (Score/Distancia: 0.6266) ---
📄 Documento: FDS 19 - KORAZA NEGRO 20380 _ 10016550 _COL (Versión 2)
📌 Sección: 8
🔖 Subtítulo:  > SECCIÓN 8: CONTROLES DE EXPOSICIÓN/PROTECCIÓN PERSONAL (continúa)
📝 Extracto: [PICTOGRAMA: Persona con gafas de seguridad y protector facial]  
| Pictograma                         | EPP                   | Observaciones               